# a2_grid

> Phase 21 (`specs/multiagent/phase21/`): A2's grids and plan. `ARMS21` and `TEAMS21` (8 teams per arm, paired on board seed), `p2Grid` (the rule-text screen: Phase 14's `scaleB` liar cells at 8 and 32 agents, original and drop-only, seeds 900-907; the originals at 900-904 are Phase 14's own requests, served by the cache), `pilotGrid21`, `A2_PLAN` (frozen and pushed before the pilot), `estimate21` (new calls and their cost: boards 1-2 once per team, P2's cached originals free), `costRule21` (the main grid's projection from the pilot's cost per call; always "report and wait"), `perCall21`, `manipulated` and `pilotCheck21` (P21-12).

In [ ]:
#| default_exp a2_grid

In [ ]:
#| hide
from nbdev.showdoc import *

## Cells

In [ ]:
#| export
import json

from explRL.a2_cols import BOARDS, failureIds, successIds
from explRL.a2_seq import TEXTS21, _key, boardCell21, historyEntry, recordLines
from explRL.colony_grid import WORLD18
from explRL.fix_grid import cellCalls, fixCell
from explRL.llm_grid import HAIKU

ARMS21 = ('none', 'shared', 'own')
TEAMS21, PILOT21, P2_SEEDS, P2_CACHED, P2_PILOT, P2_N = list(range(2100, 2108)), 2190, list(range(900, 908)), set(range(900, 905)), 990, (8, 32)
UNIT21 = dict(guards16=0.198, volunteer16=0.213, plain8=0.110, plain32=0.637)   # $ per board measured in Stages 1, 3 and Phase 14
RECORD_COST = 0.46                     # upper bound: 150 more input tokens x 480 calls x 64 record-arm boards x $0.10 per million


def boardSeed(team, b):
    return 10 * team + b


def p2Cell(N, seed, drop):
    "Phase 14's scaleB liar cell (N // 4 liars); drop: the same with dropShare (study a2drop, level 'drop|liar')."
    c = fixCell('scaleB', 'baseline', N, seed, N // 4)
    if drop:
        c = dict(c, study='a2drop', factor='a2drop', level='drop|liar', cfg=dict(c['cfg'], world=dict(c['cfg']['world'], dropShare=True)))
    return c


def p2Grid(seeds=P2_SEEDS):
    "The rule-text screen: original and drop-only at N 8 and 32, the given seeds (32 cells at 900-907)."
    return [p2Cell(N, s, d) for d in (False, True) for N in P2_N for s in seeds]


def pilotGrid21():
    "The pilot (not analysed): team 2190's own-record sequence of 6 boards, and one drop-only board at 32 agents (seed 990)."
    return dict(teams=[PILOT21], arms=['own'], p2=[p2Cell(32, P2_PILOT, True)])


def boardType(r):
    "'guards16' (boards 1-2), 'volunteer16' (boards 3-6), 'plain8' or 'plain32' (P2), from a cell or a row."
    lv, N = str(r['level']), int(r['N'])
    return ('guards16' if lv.startswith('common|') else 'volunteer16') if '|b' in lv else f'plain{N}'

## Plan, cost and the pilot's checks

`A2_PLAN` holds every frozen item of FR-B6: arms, teams, seeds, the seating rule and the record texts, the hypotheses with their tests and families, the manipulation check, the claim rules, the cost rule (report and wait; $50 expected ceiling) and the cap ($250 per calendar month). The forecasts (P21-8), the approvals and the pilot's verdict are filled at the freezes.

In [ ]:
#| export
A2_PLAN = dict(
    version=1, frozen=False, model=HAIKU, effort='low', maxTokens=8000, turns=40, memory='full', channel='blackboard',
    world=WORLD18, N=16, liars=4, honest=12, guards=4, teams=TEAMS21, pilotTeam=PILOT21, arms=list(ARMS21), boards=BOARDS,
    sharedBoards=[1, 2], boardSeed='10 * team + b', seating='a2_cols.seating(team, b): default_rng([team, b]).permutation(12), '
    'guard identities first on boards 1-2; guardIds default_rng([team, 0]); successIds default_rng([team, 99])',
    boardsOneTwo='colonyGuards (4 guards), TargetedLiar: hit the success guards, never the failure guards (P21-2)',
    boardsThreeSix='colonyVolunteer, ClueLiar (independent)', texts=TEXTS21,
    p2=dict(sizes=list(P2_N), seeds=P2_SEEDS, cached=sorted(P2_CACHED), pilotSeed=P2_PILOT, cells='scaleB baseline|liar, N // 4 liars',
            measure='suppressedHeld', also=['suppressed', 'firstWrongPerTurn', 'misinformedFirst']),
    measures=dict(guarding='turns posting a residue of a clue it holds / its turns (P21-5)',
                  contrast='per team and arm: mean over boards 3-6 of success pair mean guarding - failure pair mean guarding',
                  gateStatistic='per team: contrast(own) - contrast(shared)', dol="rl19_spec.dolIndex(rs, 4, positions=range(2, 6))"),
    hypotheses=dict(
        H21_1=dict(claim='own-record contrast > shared-record contrast', test='one-sided exact sign-flip over 8 teams', alpha=0.05, family='Gate A2'),
        H21_2=dict(claim='own-record contrast > 0', test='one-sided exact sign-flip over 8 teams, bootstrap interval', family='secondary, uncorrected'),
        H21_3=dict(claim='drop-only suppressedHeld < original, per size', test='one-sided exact sign-flip over 8 seeds', family='Holm across N 8 and 32'),
        H21_4=dict(claim='descriptive: persistence, volunteers, specialization, division of labour, welfare by arm', test='bootstrap intervals over teams')),
    holm=dict(p2=['H21_3_8', 'H21_3_32']), alpha=0.05, nboot=2000,
    manipulation=dict(teams=6, of=8, rule='both success guards have a contest on boards 1-2 followed by a correct answer'),
    claims=['Gate A2 passes only if H21-1 passes and the manipulation check passed',
            'H21-1 passing with H21-2 failing is reported as records changed guarding but not as predicted'],
    pilotGate=dict(maxInvalid=0.05), costRule=dict(ceiling=50.0, rule='report and wait'), approved=dict(pilot=None, main=None), pilotVerdict=None,
    cap=250.0, capNote='per calendar month (the user, 2026-11-09)', notMeasured=['cost without liars', 'damage won back', 'contamination across squads'],
    forecasts=dict(file='specs/multiagent/phase21/a2_predictions.json', sha256=None), exploratory=True)


def estimate21(teams=TEAMS21, arms=ARMS21, seeds=P2_SEEDS, units=None):
    """New calls and expected cost of the main grid: P1's boards 1-2 once per team, boards 3-6 per arm; P2's drop-only
    cells and the originals not in the cache (seeds 900-904 cost 0); the record lines' upper bound; total with the pilot."""
    u = units or UNIT21
    one, three = cellCalls(boardCell21(teams[0], 1, None)), cellCalls(boardCell21(teams[0], 3, 'none'))
    p2 = [c for c in p2Grid(seeds) if not (c['study'] == 'scaleB' and c['seed'] in P2_CACHED)]
    n = dict(guards16=2 * len(teams), volunteer16=4 * len(teams) * len(arms),
             plain8=sum(c['N'] == 8 for c in p2), plain32=sum(c['N'] == 32 for c in p2))
    calls = dict(guards16=one * n['guards16'], volunteer16=three * n['volunteer16'],
                 plain8=sum(cellCalls(c) for c in p2 if c['N'] == 8), plain32=sum(cellCalls(c) for c in p2 if c['N'] == 32))
    cost, pilot = {k: n[k] * u[k] for k in n}, 2 * u['guards16'] + 4 * u['volunteer16'] + u['plain32']
    main = sum(cost.values()) + RECORD_COST
    return dict(boards=n, calls=calls, cost=cost, records=RECORD_COST, pilot=pilot, main=main, total=main + pilot,
                margin=1.15 * (main + pilot), perBoard=dict(guards16=one, volunteer16=three))


def perCall21(rows):
    "$ per call by board type from measured rows (the pilot)."
    out = {}
    for r in rows:
        c, n = out.setdefault(boardType(r), [0.0, 0])
        out[boardType(r)] = [c + float(r['cost']), n + int(r['calls'])]
    return {k: c / n for k, (c, n) in out.items() if n}


def costRule21(perCall, est=None, plan=A2_PLAN):
    "The main grid's projection: its new calls per board type x the pilot's $ per call (plain8 falls back to plain32's); never 'run'."
    e = est or estimate21()
    pc = dict(perCall, plain8=perCall.get('plain8', perCall.get('plain32')))
    by = {k: e['calls'][k] * pc[k] for k in e['calls']}
    return dict(byType=by, projected=sum(by.values()), ceiling=plan['costRule']['ceiling'], verdict='ask',
                rule='report the projection to the user and wait for approval (P21-10)')


def manipulated(rows, team):
    "Both success guards of `team` have a contest on boards 1-2 that a correct answer followed (one arm's rows)."
    fol = [sum(json.loads(r['idFollowed'])[a] for r in rows if _key(r)[0] == team and _key(r)[2] <= 2) for a in successIds(team)]
    return all(f >= 1 for f in fol)


def pilotCheck21(rows, records=(), p2rows=(), plan=A2_PLAN):
    """P21-12 on the pilot: invalid replies under 5%; on boards 1-2 both success guards contest and a correct answer follows
    a success guard's contest; no lie on a failure guard's clue; the record lines as frozen. Verdict always 'ask'."""
    team, b12 = plan['pilotTeam'], [r for r in rows if _key(r)[2] <= 2]
    ids = lambda col, who: [sum(json.loads(r[col])[a] for r in b12) for a in who]
    inv = sum(float(r['invalid']) for r in list(rows) + list(p2rows)) / (sum(float(r['calls']) for r in list(rows) + list(p2rows)) or 1)
    hist = [historyEntry(r) for r in sorted(rows, key=_key)]
    lines = [tuple(map(tuple, x['lines'])) == recordLines('none' if x['board'] <= 2 else x['level'].split('|')[0], hist[:x['board'] - 1], x['board'], x['team'])
             for x in records]
    checks = dict(boards=sorted(_key(r)[2] for r in rows) == list(range(1, plan['boards'] + 1)), invalid=inv < plan['pilotGate']['maxInvalid'],
                  successContest=all(n >= 1 for n in ids('idContests', successIds(team))),
                  successFollowed=sum(ids('idFollowed', successIds(team))) >= 1,
                  noFailureLies=all(int(r['liarOnFailure']) == 0 for r in b12),
                  records=bool(records) and all(lines) and plan['texts'] == TEXTS21)
    return dict(ok=all(checks.values()), checks=checks, reasons=[k for k, v in checks.items() if not v], invalidShare=inv,
                perCall=perCall21(list(rows) + list(p2rows)), failureIds=failureIds(team), verdict='ask')

## Tests

In [ ]:
# test_p2grid (FR-B5, AC-2): the originals are Phase 14's scaleB liar cells (same request keys: the cache serves 900-904);
# the drop-only cells differ by dropShare alone; 32 cells
from explRL.fix_grid import requestKey
from explRL.scale_grid import scaleGrid
from explRL.a2_grid import *
from explRL.a2_grid import _key

g = p2Grid()
assert len(g) == 32 and {c['N'] for c in g} == {8, 32} and sorted({c['seed'] for c in g}) == P2_SEEDS
phase14 = {requestKey(c) for c in scaleGrid() if c['level'] == 'baseline|liar' and c['N'] in P2_N}
orig = [c for c in g if c['study'] == 'scaleB']
assert {requestKey(c) for c in orig if c['seed'] in P2_CACHED} == phase14 and len(phase14) == 10
assert all(c['cfg']['nLiar'] == c['N'] // 4 and c['cfg']['maxTokens'] == 8000 for c in g)
for c in g:
    if c['study'] == 'a2drop':
        o = p2Cell(c['N'], c['seed'], False)
        assert c['level'] == 'drop|liar' and c['cfg']['world'] == dict(o['cfg']['world'], dropShare=True)
        assert {k: v for k, v in c['cfg'].items() if k != 'world'} == {k: v for k, v in o['cfg'].items() if k != 'world'}
assert boardType(g[0]) == 'plain8' and boardType(boardCell21(2100, 1, None)) == 'guards16' and boardType(boardCell21(2100, 3, 'own', [dict(board=1, contests=[0] * 12, followed=[0] * 12), dict(board=2, contests=[0] * 12, followed=[0] * 12)])) == 'volunteer16'

In [ ]:
# test_grid21 (FR-B7, T3's counts): 480 calls per 16-agent board, 240 and 960 per P2 board; boards 1-2 once per team;
# P2's originals at 900-904 cost 0; the design's $34.19 total; costRule21 never runs on its own; the plan's items
e = estimate21()
assert e['perBoard'] == dict(guards16=480, volunteer16=480)
assert e['boards'] == dict(guards16=16, volunteer16=96, plain8=11, plain32=11), e['boards']
assert e['calls'] == dict(guards16=16 * 480, volunteer16=96 * 480, plain8=11 * 240, plain32=11 * 960)
assert abs(e['total'] - 34.19) < 0.015 and round(e['main'], 2) == 32.29 and round(e['pilot'], 2) == 1.89 and 39 <= e['margin'] <= 39.5, e   # the design adds rounded lines: $34.19
cr = costRule21(dict(guards16=0.198 / 480, volunteer16=0.213 / 480, plain32=0.637 / 960))
assert cr['verdict'] == 'ask' and cr['ceiling'] == 50.0 and abs(cr['projected'] - (16 * .198 + 96 * .213 + 11 * .637 + 11 * .637 / 4)) < 1e-9
for k in ('arms', 'teams', 'texts', 'hypotheses', 'manipulation', 'claims', 'costRule', 'cap', 'capNote', 'forecasts', 'approved', 'p2', 'maxTokens'):
    assert k in A2_PLAN, k
assert A2_PLAN['cap'] == 250.0 and A2_PLAN['capNote'] == 'per calendar month (the user, 2026-11-09)' and A2_PLAN['maxTokens'] == 8000
assert pilotGrid21()['teams'] == [2190] and pilotGrid21()['p2'][0]['N'] == 32 and pilotGrid21()['p2'][0]['seed'] == 990

In [ ]:
# pilotCheck21 and manipulated on hand-built rows (P21-12): passes when both success guards contest, one follow, no lie on
# failure-guard clues and the records match; each failure is named
def fakeRows(team, arm='own', cont=1, fol=1, fail=0, inv=0):
    S, rows = successIds(team), []
    for b in range(1, 7):
        c, f = [0] * 12, [0] * 12
        for a in S:
            c[a], f[a] = (cont, fol) if b <= 2 else (0, 0)
        rows.append(dict(seqId=team, arm=arm, pos=b - 1, level=('common' if b <= 2 else arm) + f'|b{b}', N=16, idContests=json.dumps(c),
                         idFollowed=json.dumps(f), liarOnFailure=fail if b <= 2 else 0, invalid=inv, calls=480, cost=0.2))
    return rows

rows = fakeRows(2190)
hist = [historyEntry(r) for r in rows]
recs = [dict(team=2190, board=b, level=('common' if b <= 2 else 'own') + f'|b{b}',
             lines=[list(x) for x in recordLines('none' if b <= 2 else 'own', hist[:b - 1], b, 2190)]) for b in range(1, 7)]
P = pilotCheck21(rows, recs)
assert P['ok'] and P['verdict'] == 'ask' and abs(P['perCall']['guards16'] - 0.2 / 480) < 1e-12, P
assert pilotCheck21(rows, []) ['reasons'] == ['records']
assert pilotCheck21(fakeRows(2190, cont=0, fol=0), recs)['reasons'][:2] == ['successContest', 'successFollowed']
assert 'noFailureLies' in pilotCheck21(fakeRows(2190, fail=1), recs)['reasons']
assert 'invalid' in pilotCheck21(fakeRows(2190, inv=30), recs)['reasons']
bad = [dict(x, lines=[[4, 'x']] + x['lines'][1:]) if x['board'] == 4 else x for x in recs]
assert pilotCheck21(rows, bad)['reasons'] == ['records']
assert manipulated(fakeRows(2100), 2100) and not manipulated(fakeRows(2100, fol=0), 2100)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()